# Coefficient-space arterial spectral resonance: full computational study

This notebook is the **thin orchestration and plotting interface** for the `arterial_spectral_cascade` package. The Mathematical Model and Solver Design are implemented in the package; this notebook configures explicit morphology/coefficient cases, persistent storage, execution, and figure regeneration.

The default mode is `FULL_STUDY`. A full study will not start until explicit disease cases with morphology parameters and signed coefficient sensitivities are supplied.


## 1. Mathematical Model

The solved reduced-order equation is

$$
a_s+a a_\xi+b(\xi)a_{\xi\xi\xi}+g(\xi)\Lambda a=0,
\qquad
\Lambda=(-\partial_\xi^2)^{1/2}.
$$

The Womersley-dependent baselines are inherited from the published *Physics of Fluids* model,

$$
b_0(\mathrm{Wo})=b_{\mathrm{ref}}\mathrm{Wo}^{-2},
$$

$$
g_0(\mathrm{Wo})=g_{\mathrm{ref}}\left(1+\frac{C_g}{\mathrm{Wo}}\right).
$$

Disease enters only through a normalized periodic morphology field $0\leq\Psi_D\leq1$ and signed effective coefficient sensitivities $\chi_b,\chi_g$:

$$
b(\xi)=b_0(\mathrm{Wo})[1+\varepsilon_b\cos(q\xi)+\chi_b\Psi_D(\xi)],
$$

$$
g(\xi)=g_0(\mathrm{Wo})[1+\varepsilon_g\cos(q\xi)+\chi_g\Psi_D(\xi)].
$$

For primary disease-only calculations, $\varepsilon_b=\varepsilon_g=0$. The package does not infer $\chi_b$ or $\chi_g$ from a radius, local Womersley number, clinical label, or anatomical sign convention.


## 2. Solver Design

The coefficient fields are decomposed exactly as

$$
b=\bar b+\widetilde b,
\qquad
g=\bar g+\widetilde g.
$$

The diagonal mean operator

$$
L_0(k)=-i\bar b k^3-\bar g|k|
$$

is advanced analytically by ETDRK4. The nonlinear and heterogeneous residual is

$$
\mathcal F(a)
=-\frac12\partial_\xi(a^2)
-\widetilde b\,a_{\xi\xi\xi}
-\widetilde g\,\Lambda a.
$$

The implementation uses the signed Fourier convention of the Solver Design, symmetric two-thirds de-aliasing, cancellation-safe $\varphi_j$ functions, preservation of the dynamically generated zero mode, and the exact heterogeneous integral balances as numerical diagnostics.


## 3. Disease morphology input contract

The canonical Mathematical Model classes are:

- `DL`: a single localized smooth periodic field $\Psi_L$;
- `DM`: a normalized multiple-lesion field $\Psi_M$;
- `DR`: a distributed irregular field $\Psi_R$ or a supplied geometry-derived sampled field;
- `MM`: the exact matched-mean control generated from each heterogeneous case.

A geometry-derived `DR` field must be normalized, sampled on the solver grid, accompanied by provenance, and supplied with an externally justified or parametrically varied pair $(\chi_b,\chi_g)$. The package intentionally contains no default clinical severity-to-coefficient mapping.


## 4. Install the local package

The archive should be extracted so that `arterial_spectral_cascade_package` is present either in the current working directory or at `/content/arterial_spectral_cascade_package` in Google Colab. This cell installs that local source tree in editable mode and exposes its `src` directory immediately to the current Python kernel.

In [ ]:
from pathlib import Path
import importlib, os, sys, subprocess

candidates = [
    Path.cwd(),
    Path.cwd() / "arterial_spectral_cascade_package",
    Path("/content/arterial_spectral_cascade_package"),
]
PACKAGE_ROOT = next((p.resolve() for p in candidates if (p / "pyproject.toml").exists()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError(
        "Could not locate arterial_spectral_cascade_package. Extract the supplied zip into "
        "the current directory or /content, then rerun this cell."
    )

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "--no-deps", "-e", str(PACKAGE_ROOT)])

# Editable-install metadata may not become visible to an already-running notebook kernel.
PACKAGE_SRC = str((PACKAGE_ROOT / "src").resolve())
if PACKAGE_SRC not in sys.path:
    sys.path.insert(0, PACKAGE_SRC)
importlib.invalidate_caches()

# Verify the import in this same kernel so a first-run Colab failure occurs here, not later.
import arterial_spectral_cascade as _asc_import_check
print(f"Installed local package from: {PACKAGE_ROOT}")
print(f"Current-kernel import verified: {_asc_import_check.__file__}")

## 5. Import the scientific interface and inspect the morphology classes

The available user-facing modes are `QUICK_CHECK`, `VERIFICATION`, `PARAMETER_SELECTION`, `FULL_STUDY`, and `FIGURES`.

Before `FULL_STUDY`, populate `STUDY_CONFIG["DISEASE_CASES"]` with explicit coefficient-space cases. The commented template below shows the required structure without assigning unsupported clinical sensitivities.


In [ ]:
from copy import deepcopy
from arterial_spectral_cascade import __version__
from arterial_spectral_cascade.config import default_study_config
from arterial_spectral_cascade.study import morphology_class_table, configured_case_table, configured_root, run_study_mode
from arterial_spectral_cascade.storage import init_project_paths
from arterial_spectral_cascade.plotting import regenerate_available_figures

STUDY_CONFIG = default_study_config()
STUDY_CONFIG["RUN_MODE"] = "FULL_STUDY"

# Example structure only; replace the ellipses with scientifically justified values.
# STUDY_CONFIG["DISEASE_CASES"] = (
#     {
#         "case_id": "localized_case_1",
#         "case_class": "DL",
#         "xi_c": 2*3.141592653589793,
#         "w": ...,
#         "p": 1,
#         "chi_b": ...,
#         "chi_g": ...,
#     },
# )

print("arterial-spectral-cascade version:", __version__)
print("Run mode:", STUDY_CONFIG["RUN_MODE"])
print("Configured disease cases:", len(STUDY_CONFIG["DISEASE_CASES"]))
morphology_class_table()


## 6. Persistent Google Drive storage

In Colab, the default study directory is `/content/drive/MyDrive/PoF_ArterialSpectralCascade`. All full-resolution trajectories are restartable. Case metadata, checkpoints, verification reports, result archives, tables, figures, and logs are written in separate directories.

To use a different location, set `STUDY_CONFIG["PROJECT_ROOT"]` before running the next cell.

In [ ]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and STUDY_CONFIG.get("MOUNT_DRIVE", True):
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT_ROOT = configured_root(STUDY_CONFIG)
PATHS = init_project_paths(PROJECT_ROOT)
print("Persistent study directory:", PATHS.root)

## 7. Numerical settings and acceptance logic

The default numerical starting point is $N=512$ and $\Delta s=2\times10^{-4}$. `PARAMETER_SELECTION` evaluates demanding admissible coefficient-space cases for each configured morphology class and determines common numerical settings before the main calculations.

The parent-reference procedure separates numerical recovery from historical resonance-topology comparison. The Solver Design parent trajectories must complete and pass the independent verification requirements. Whether the resulting discrete baseline reproduces a previously reported peak location is retained only as a reference diagnostic and is not an acceptance criterion.


## 8. Result set

**R1 — Morphology to heterogeneous spectral coupling.** The solver records $\Psi_D(\xi)$, $b(\xi)$, $g(\xi)$, coefficient spectra, and the heterogeneous coupling matrix $H_{\ell n}$.

**R2 — Coefficient-space resonance response.** Each configured disease morphology/sensitivity case is resolved across Womersley number with local refinement that does not assume a prescribed resonance topology.

**R3 — Heterogeneous versus matched-mean response.** Every heterogeneous disease calculation has the exact matched-mean control,

$$
\Delta R(s)=R_{\mathrm{het}}(s)-R_{\mathrm{mm}}(s),
$$

$$
D_2(s)=\frac{\|a_{\mathrm{het}}-a_{\mathrm{mm}}\|_{L^2}}{\max(\|a_{\mathrm{mm}}\|_{L^2},\epsilon_{\mathrm{mach}})}.
$$

**R4 — Mechanistic modal-energy analysis.** Selected cases resolve the nonlinear, heterogeneous-dispersion, heterogeneous-damping, and mean-damping contributions to the modal energy rate.

**R5 — Morphology-scale selectivity.** When explicit scale factors are configured for analytical localized/multiple morphologies, the package evaluates the dependence of the heterogeneous-minus-matched-mean response on morphology scale. Geometry-derived sampled fields are never rescaled silently.


## 9. Execute the study

For a complete run, first supply explicit admissible entries in `STUDY_CONFIG["DISEASE_CASES"]`. The sequence is restartable:

`QUICK_CHECK` $\rightarrow$ `VERIFICATION` $\rightarrow$ `PARAMETER_SELECTION` $\rightarrow$ primary coefficient-morphology calculations $\rightarrow$ mechanism cases $\rightarrow$ optional morphology-scale study $\rightarrow$ publication figures.

A compatible completed step is reused from persistent storage. Invalid or incompatible verification records are not silently accepted.


In [ ]:
STUDY_REPORT = run_study_mode(PATHS, STUDY_CONFIG, progress=True)
STUDY_REPORT

## 10. Physics of Fluids publication figures

The plotting layer uses the locked AIP/*Physics of Fluids* template. Figures are prepared at final publication dimensions, use at least 8-pt text and 0.5-pt lines, distinguish curves by line style and/or marker as well as color, omit internal panel titles, and label multipart panels `(a)`, `(b)`, and so forth.

Each figure is written as vector PDF, vector SVG, 600-dpi PNG, and an alt-text sidecar. Figures are regenerated only from persisted validated result archives.

In [ ]:
FIGURE_FILES = regenerate_available_figures(PATHS, STUDY_CONFIG)
print(f"Publication figure files: {len(FIGURE_FILES)}")
for f in FIGURE_FILES:
    print(f)

## 11. Scope

The calculations determine how bounded coefficient-space disease morphology alters spectral coupling and resonance within the Mathematical Model and Solver Design. They do not infer a disease-to-coefficient constitutive law and do not predict plaque progression, rupture, thrombosis, wall stress, wall shear stress, treatment efficacy, or patient outcome.
